In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

from matplotlib.gridspec import GridSpec
#from ipywidgets import interact, FloatSlider, FloatLogSlider
import warnings
warnings.filterwarnings('ignore')
import os

from scipy.ndimage import gaussian_filter


import visualize_correct_z as vis
import matplotlib.colors as mcolors

from scipy.signal import find_peaks

import scipy.integrate as integrate


# visualize paraview linecuts

In [ ]:
def get_slice(sample, coord, date, data_dir):
    """
    Returns a dict of all parameters/paths needed to load a single
    (sample, slice) combination.
    """

    cfg = dict(
        sample=sample,
        coord=coord
    )

    print(cfg)

    date = date

    if sample == '665':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 51 #nm
    elif sample == '270':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 66 #nm
    elif sample == '452':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 33 #nm
    elif sample == '105':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 45 #nm 
    elif sample == '665_magconst':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 51 #nm
    elif sample == '270_magconst':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 66 #nm
    elif sample == '452_magconst':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 33 #nm
    elif sample == '105_magconst':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 45 #nm 

    elif sample == '452_orig':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 33 #nm
    elif sample == '105_orig':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 45 #nm 


    else:
        raise ValueError(f"Unrecognized sample '{sample}'")

    print(f"{sample}_100_{coord}_center_slice_{date}.csv")
    file_path = os.path.join(data_dir, f"{sample}_100_{coord}_center_slice_{date}.csv")
    
    # file_path = os.path.join(data_dir, f"{sample}_{coord}_center_slice_{date}.csv")
    data = np.genfromtxt(file_path, delimiter=",", names=True)

    # if coord == 'y':
    #     coord1 = data['Points0']
    #     coord2 = data['Points2']
    #     mz = data['magnetization2']
    #     mx = data['magnetization1'] ############# I changed this based on observation, may be wrong
    #     my = data['magnetization0']

    # elif coord == 'x':
    #     coord1 = data['Points1']
    #     coord2 = data['Points2']
    #     mz = data['magnetization2']
    #     mx = data['magnetization1'] ############# I changed this based on observation, may be wrong
    #     my = data['magnetization0']

    if coord == 'y':
        coord1 = data['Points0']
        coord2 = data['Points2']
        coord_name = data['Points1']
        mz = data['M2']
        mx = data['M0'] 
        my = data['M1']

    elif coord == 'x':
        coord1 = data['Points1']
        coord2 = data['Points2']
        coord_name = data['Points0']
        mz = data['M2']
        mx = data['M1'] 
        my = data['M0']

    eps = 1e-12
    mnorm = np.sqrt(mx**2 + my**2 + mz**2) # + eps # same as r!! 
    mxn = mx / mnorm
    myn = my / mnorm
    mzn = mz / mnorm

    theta = np.arccos(np.clip(mzn, -1.0, 1.0))

    cfg['slice'] = data
    cfg['x'] = coord1
    cfg['z'] = coord2
    cfg['mx'] = mx
    cfg['my'] = my
    cfg['mz'] = mz
    cfg['mxn'] = mxn
    cfg['myn'] = myn
    cfg['mzn'] = mzn
    cfg['theta'] = theta
    cfg['mnorm'] = mnorm

    return cfg


### User set up

In [ ]:
### User set up
# -----------------------------------------------------------------------
# Scan To Analyze
# -----------------------------------------------------------------------

# sample_ids = ['665', '270', '452', '105']
# dates = ['11JULY26','11JULY26', '11JULY26', '11JULY26']

# sample_ids = ['270', '270_magconst', '452', '452_magconst']
# dates = ['10AUG26', '10AUG26', '10AUG26', '10AUG26']

# sample_ids = ['270', '270_magconst']
# dates = ['10AUG26', '10AUG26']

# sample_ids = ['452', '452_magconst']
# dates = ['10AUG26', '10AUG26']

# sample_ids = ['270', '452']
# dates = ['11AUG26', '11AUG26']

# sample_ids = ['452']
# dates = ['11AUG26']

sample_ids = ['452', '452_magconst', '452_orig']
dates = ['14AUG26', '14AUG26', '11AUG26']

# sample_ids = ['270', '270_magconst']
# dates = ['14AUG26', '14AUG26']



#base_path = '/homes/emma/notebooks/RawData/{sample}/vectors/'
#base_path = r'C:\Users\emmab\Projects\Vector_Tomography\665_z_center_slice_1JULY26.csv'
base_path = ''

field_pairs = [('mz', 'mzn'), ('my', 'myn'), ('mx', 'mxn')]

# -----------------------------------------------------------------------
# SMOOTHING FLAG -- set this once, here, and it applies to everything
# downstream (both normalized AND un-normalized grids, for every sample).
# -----------------------------------------------------------------------
SMOOTH_DATA = True     # <-- set False to skip Gaussian smoothing entirely
SMOOTH_SIGMA = 2       # <-- Gaussian filter sigma (applied along x only)
how_many_slices = 9 #5 # (steps in percentage)

samples = {}
for sample, date in zip(sample_ids, dates):
    samples[sample] = {
        'y': get_slice(sample, 'y', date, base_path.format(sample=sample)),
        'x': get_slice(sample, 'x', date, base_path.format(sample=sample)),
        # 'z': get_slice(sample, 'z', date, base_path.format(sample=sample)),
    }

# print(samples['665']['y'].keys())

# z = samples['665']['y']['z']
# print(z.shape)
# print(np.min(z))
# print(np.max(z))

# x = samples['665']['y']['x']
# mz = samples['665']['y']['mz']
# Mmag = samples['665']['y']['mnorm']

# plt.plot(x,z)
# plt.xlabel('y')
# plt.ylabel('z')
# plt.show()

# plt.plot(x,mz)
# plt.xlabel('y')
# plt.ylabel('mz')
# plt.show()

# plt.plot(x,Mmag)
# plt.xlabel('y')
# plt.ylabel('Mmag')
# plt.show()

# Every cfg dict built here now stores BOTH the un-normalized ('{field}_grid')
# and normalized ('{field_n}_grid') grid for every field pair, and -- if
# SMOOTH_DATA is True -- both have already been Gaussian-smoothed (sigma =
# SMOOTH_SIGMA, along x only) before anything else touches them.
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        coords[coord] = vis.build_grids(
            data_dict, field_pairs, n_show=how_many_slices, xrange = None,
            smooth=False, sigma=SMOOTH_SIGMA
        )


# -----------------------------------------------------------------------
# Plot set up
# -----------------------------------------------------------------------


plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'lines.linewidth': 2.2,
    'axes.grid': False,
    'grid.alpha': 0.3,
})


In [ ]:
def find_tilt_angle(y, z):
    slope, intercept = np.polyfit(y, z, 1)
    angle = np.arctan(slope)
    return angle

def rotate_to_flat(y, z, angle=None):
    """
    Rotates a (y, z) point cloud to remove a linear tilt, so the data
    becomes flat/horizontal. If angle is None, it's auto-detected via a
    linear fit through the point cloud.
    Returns y_flat, z_flat, and the angle used (in case you want to reuse
    the same angle on a companion dataset, e.g. a different field slice
    from the same tilted geometry).
    """
    if angle is None:
        angle = find_tilt_angle(y, z)

    y_center = y.mean()
    z_center = z.mean()

    y_shifted = y - y_center
    z_shifted = z - z_center

    cos_a, sin_a = np.cos(-angle), np.sin(-angle)
    y_flat = y_shifted * cos_a - z_shifted * sin_a
    z_flat = y_shifted * sin_a + z_shifted * cos_a

    y_flat += y_center
    z_flat += z_center

    return y_flat, z_flat, angle



for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        x_arr = data_dict['x']
        z_arr = data_dict['z']

        calc_tilt = find_tilt_angle(x_arr, z_arr)
        print(f'{sample} {coord}: tilt = {np.degrees(calc_tilt):.2f}°')

        if abs(calc_tilt) > np.radians(1):  # only correct tilts over ~1 degree
            x_flat, z_flat, angle = rotate_to_flat(x_arr, z_arr, angle=calc_tilt)
            print(f"  Tilt angle removed: {np.degrees(angle):.2f}°")

            # save the corrected coordinates back into the dictionary
            data_dict['x_raw'] = x_arr  # preserve original, same pattern as your smoothing/_raw keys
            data_dict['z_raw'] = z_arr
            data_dict['x'] = x_flat
            data_dict['z'] = z_flat
            data_dict['tilt_angle'] = angle

            plt.figure(figsize=(10, 5))
            plt.scatter(x_flat, z_flat, s=1)
            plt.xlabel('x')
            plt.ylabel('z')
            plt.title(f'{sample} {coord} — de-tilted slice ({np.degrees(angle):.1f}° removed)')
            plt.show()
        else:
            data_dict['tilt_angle'] = 0.0

# full profile

In [ ]:
#print(samples['270']['x'].keys())
def plot_magnitude(cfg, z_indices=None):
    """
    Line-profile version: pick a handful of specific voxel rows (z_indices)
    to overlay, rather than showing the full heatmap.
    """
    unique_x = cfg['unique_x']
    z_grid = cfg['z_grid']

    
    if z_indices is None:
        # default: 5 evenly spaced rows across the available depth
        z_indices = np.linspace(0, cfg['n_rows'] - 1, 5).astype(int)

    colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    fig, ax = plt.subplots(figsize=(49, 7))

    for c, row in zip(colors, z_indices):
        ax.plot(unique_x, cfg['M_mag'][row, :], color=c, lw=1.5,
                    label=fr'z={z_grid[row]:.1f}')
        ax.set_ylabel(r'$M_{mag}$')
        ax.set_xlabel('x (pix)')
        ax.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper left')
        ax.set_title(f"Magnitude \n full {cfg['coord']}-slice profile \n {cfg['sample']}")

plt.tight_layout()
plt.show()

for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        plot_magnitude(samples[sample][coord])


# select one wall

In [ ]:
# crop1_ranges = {
#     '665': {'x': (495, 508), 'y': (485, 515)},
#     '270': {'x': (590, 615), 'y': (575, 590)},
#     '452': {'x': (515, 525), 'y': (415, 435)},
#     '105': {'x': (503, 520), 'y': (540, 555)},
# }

# crop1_ranges = {
#     '665': {'x': (495, 508), 'y': (485, 515)},
#     '270': {'x': (584, 596), 'y': (575, 590)},
#     '452': {'x': (512, 525), 'y': (819, 833)},
#     '105': {'x': (503, 520), 'y': (540, 555)},
# }


# crop1_ranges = {
#     '270': {'x': (30, 55), 'y': (18, 30)},
#     '270_magconst': {'x': (30, 55), 'y': (18, 30)},
#     '452': {'x': (80, 96), 'y': (35, 59)},
#     '452_magconst': {'x': (80,96), 'y': (35,59)},
# }

# crop1_ranges = {
#     '270': {'x': (80, 100), 'y': (80, 100)},
#     '452': {'x': (784, 794), 'y': (723, 745)},
# }

crop1_ranges = {
    '270': {'x': (63, 75), 'y': (0, 15)},
    '270_magconst': {'x': (63, 75), 'y': (0, 15)},
    '452': {'x': (27, 43), 'y': (27, 43)},
    '452_magconst': {'x': (27, 43), 'y': (27, 43)},
    '452_orig': {'x': (727, 743), 'y': (727, 743)},
}



crop1 = {}
for sample, coords in samples.items():
    crop1[sample] = {}
    for coord, data_dict in coords.items():
        xmin, xmax = crop1_ranges[sample][coord]
        crop1[sample][coord] = {
            'dw': vis.build_grids(
                data_dict, field_pairs, n_show=how_many_slices,
                xrange = (xmin, xmax),
                smooth=SMOOTH_DATA, sigma=SMOOTH_SIGMA
            ),
            'xmin': xmin,
            'xmax': xmax,
        }


In [ ]:
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        # if sample == '270_magconst': # and coord == 'x':
        #     plot_grad = True
        # else:
        plot_grad = True

        crop1[sample][coord]['dw'] = vis.gradient(crop1[sample][coord]['dw'], field_pairs, normalized = True, plot=plot_grad)
        vis.gradient(samples[sample][coord], field_pairs, normalized = True, plot=False)


# for sample, coords in samples.items():
#     for coord, data_dict in coords.items():
#         # if sample == '270_magconst': # and coord == 'x':
#         #     plot_grad = True
#         # else:
#         plot_grad = True

#         vis.gradient(samples[sample][coord], field_pairs, normalized = True, plot=plot_grad)

# Calculate Psi

In [ ]:
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.calc_psi(crop1[sample][coord]['dw'], normalized =  True, plot=True)

### also calculate psi for the full (uncropped) stack, for the final
### normalized-vs-un-normalized Neel/Bloch comparison further down
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.calc_psi(samples[sample][coord], normalized =  True, plot=False)


In [ ]:
# def cart2sph(cfg, eps=1e-12):
#     """
#     Convert Cartesian vector components (mx, my, mz) to spherical
#     coordinates (r, theta, phi), following the convention:

#         r     = sqrt(x^2 + y^2 + z^2)
#         theta = arccos(z / r)                        (polar angle, 0..pi)
#         phi   = sgn(y) * arccos( x / sqrt(x^2+y^2) )  (azimuth, -pi..pi)

#     with the piecewise edge cases spelled out in the reference image
#     (z == 0, r == 0, x == 0, x == y == 0, etc.).

#     Inputs can be scalars or numpy arrays of any (matching) shape.

#     Returns
#     -------
#     cfg : dict, with 'r', 'theta_2', 'psi' added (same shape as inputs,
#         with np.nan used for the "undefined" cases in the reference).
#     """
#     # NOTE: fixed a naming inconsistency here -- you had 'mx_grid' paired
#     # with 'myn_grid' / 'mzn_grid' (normalized y, z but un-normalized x).
#     # Using the normalized triple consistently so r == 1 everywhere except
#     # at masked-out / zero voxels.
#     mx = cfg['mxn_grid']
#     my = cfg['myn_grid']
#     mz = cfg['mzn_grid']

#     mx = np.asarray(mx, dtype=float)
#     my = np.asarray(my, dtype=float)
#     mz = np.asarray(mz, dtype=float)

#     rho2 = mx**2 + my**2          # x^2 + y^2
#     rho = np.sqrt(rho2)           # sqrt(x^2 + y^2)
#     r = np.sqrt(rho2 + mz**2)     # full radius

#     origin = r < eps              # x == y == z == 0

#     # ---------------- theta ----------------
#     with np.errstate(invalid='ignore', divide='ignore'):
#         theta = np.arccos(np.clip(mz / np.where(origin, np.nan, r), -1.0, 1.0))

#     z_zero_rho_nonzero = (np.abs(mz) < eps) & (rho >= eps)
#     theta = np.where(z_zero_rho_nonzero, np.pi / 2.0, theta)
#     theta = np.where(origin, np.nan, theta)

#     # ---------------- phi (psi) ----------------
#     x_zero = np.abs(mx) < eps
#     y_zero = np.abs(my) < eps
#     rho_zero = rho < eps  # x == y == 0

#     with np.errstate(invalid='ignore', divide='ignore'):
#         base = np.arccos(np.clip(mx / np.where(rho_zero, np.nan, rho), -1.0, 1.0))

#     sgn_y = np.sign(my)
#     phi = sgn_y * base

#     phi = np.where(x_zero & (my > 0), np.pi / 2.0, phi)
#     phi = np.where(x_zero & (my < 0), -np.pi / 2.0, phi)
#     phi = np.where(x_zero & y_zero, np.nan, phi)

#     x_neg_y_zero = (mx < 0) & y_zero
#     phi = np.where(x_neg_y_zero, np.pi, phi)

#     cfg['r'] = r
#     cfg['theta_2'] = theta
#     cfg['psi'] = phi

#     return cfg


# def plot_psi2(cfg, n_bins=11, z_indices=None):
#     """Plots r, theta, and psi (all in radians -- no /pi scaling) computed
#     from Mx, My, Mz. Only the psi panel gets the wall-fraction color bands,
#     since r and theta aren't naturally binned on a [-pi, pi] scale."""

#     fig, ax = plt.subplots(1, 3, figsize=(25, 8))
#     x_data = cfg['unique_x'] - cfg['unique_x'].mean()

#     # bin edges now live in radians (psi's natural range), not units of pi
#     bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
#     cmap = plt.cm.twilight_shifted
#     bin_colors = cmap(np.linspace(0, 1, n_bins))

#     z_grid = cfg['z_grid']

#     if z_indices is None:
#         z_indices = np.linspace(0, cfg['n_rows'] - 1, cfg['n_rows']).astype(int)
#     colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

#     for c, j in zip(colors, z_indices):
#         # plotted directly in their native units -- no /np.pi scaling.
#         # r is a magnitude (not an angle), so it was never meant to be
#         # divided by pi in the first place.
#         ax[0].plot(x_data, cfg['r'][j, :], color=c, lw=1.5,
#                    marker='o', label=fr'r, z voxel ={z_grid[j]:.1f}')
#         ax[1].plot(x_data, cfg['theta_2'][j, :], color=c, lw=1.5,
#                    marker='o', label=fr'θ, z voxel ={z_grid[j]:.1f}')
#         ax[2].plot(x_data, cfg['psi'][j, :], color=c, lw=1.5,
#                    marker='o', label=fr'ψ, z voxel ={z_grid[j]:.1f}')

#     # color bands only on the psi panel (ax[2])
#     for k in range(n_bins):
#         ax[2].axhspan(bin_edges[k], bin_edges[k + 1], color=bin_colors[k],
#                       alpha=0.15, zorder=0)

#     ax[0].set_ylabel(r'r')
#     ax[0].set_xlabel('x (pix)')
#     ax[0].set_title(f"Calculated r per x \n {cfg['sample']} {cfg['coord']}-slice")

#     ax[1].set_ylabel(r'θ (rad)')
#     ax[1].set_xlabel('x (pix)')
#     ax[1].set_ylim(0, np.pi)
#     ax[1].set_title(f"Calculated θ per x \n {cfg['sample']} {cfg['coord']}-slice")

#     ax[2].set_ylabel(r'ψ (rad)')
#     ax[2].set_xlabel('x (pix)')
#     ax[2].set_ylim(-np.pi, np.pi)
#     ax[2].axhline(0, color='cyan', label='Néel wall')
#     ax[2].axhline(np.pi / 2, color='magenta', label='Bloch wall')
#     ax[2].axhline(-np.pi / 2, color='magenta')
#     ax[2].set_title(f"Calculated ψ per x \n {cfg['sample']} {cfg['coord']}-slice")

#     plt.tight_layout()
#     plt.show()


# # for sample, coords in samples.items():
# #     for coord, data_dict in coords.items():
# #         cart2sph(crop1[sample][coord]['dw'])
# #         plot_psi2(crop1[sample][coord]['dw'], n_bins=11, z_indices=None)


# Color coordinate

In [ ]:
def plot_psi_bin_offset_w_highlight(cfg, select_psi='', n_bins=11, offset=1.5, psi_tol=0.05, normalized=True, z_indices = None):
    psi_key = select_psi + 'psi_grid'
    if psi_key not in cfg:
        cfg = vis.calc_psi(cfg)

    suffix = '' if normalized else '_raw'

    if z_indices is None:
        z_indices = np.linspace(0, cfg['n_rows'] - 1, cfg['n_rows']).astype(int)
    colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_key = select_psi + f'psi{suffix}_grid'
    psi_over_pi = cfg[psi_key] / np.pi

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    z_grid = cfg['z_grid']

    if normalized:
        x_k, y_k, z_k = 'mxn_grid', 'myn_grid', 'mzn_grid'
    else:
        x_k, y_k, z_k = 'mx_grid', 'my_grid', 'mz_grid'

    # fig, ax = plt.subplots(1, 3, figsize=(26, 8))
    fig, ax = plt.subplots(3, 1, figsize=(7, 15))

    for c, j in zip(colors, z_indices):
        psi_slice = psi_over_pi[j, :]
        shift = j * offset

        mzn = cfg[z_k][j, :] + shift
        myn = cfg[y_k][j, :] + shift
        mxn = cfg[x_k][j, :] + shift

        ax[0].plot(x_data, mzn, c=c, linewidth=1)
        ax[1].plot(x_data, myn, c=c, linewidth=1)
        ax[2].plot(x_data, mxn, c=c, linewidth=1)

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx].copy()

        neel_mask = (np.abs(psi_slice) <= psi_tol) | (np.isclose(np.abs(psi_slice), 1, atol=psi_tol))
        bloch_mask = np.isclose(np.abs(psi_slice), 0.5, atol=psi_tol)
        point_colors[neel_mask] = mcolors.to_rgba('cyan')
        point_colors[bloch_mask] = mcolors.to_rgba('magenta')

        ax[0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
        ax[1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
        ax[2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

        for a in ax.flat:
            a.axhline(shift, color=c, lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'z voxel ={z_grid[j]:.1f}', xy=(0.22, shift - 0.4),
                        xycoords=('axes fraction', 'data'),
                        xytext=(-8, 0), textcoords='offset points',
                        ha='right', va='center', fontsize=10, color=c)
        

    neel_pct_overall = cfg.get(f'neel_pct_overall{suffix}')
    bloch_pct_overall = cfg.get(f'bloch_pct_overall{suffix}')
    neel_pct_per_eta = cfg.get(f'neel_pct_per_eta{suffix}')
    bloch_pct_per_eta = cfg.get(f'bloch_pct_per_eta{suffix}')

    ax[0].set_ylabel(r'$M_{z}$')
    ax[1].set_ylabel(r'$M_{y}$')
    ax[2].set_ylabel(r'$M_{x}$')
    for a in ax:
        a.set_xlabel('x (pix)')
        a.set_yticks([])



    if neel_pct_overall is not None:
        per_eta_handles = [
            plt.Line2D([0], [0], color=c,
                       label=fr'z voxel ={z_grid[j]:.1f}: Néel {neel_pct_per_eta[j]:.1f}% vs. Bloch {bloch_pct_per_eta[j]:.1f}%')
            for c, j in zip(colors, z_indices)
        ]
        legend_handles = [
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='cyan', markersize=10,
                       label=fr'Néel-like (ψ≈0,±1) — overall: {neel_pct_overall:.1f}%'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='magenta', markersize=10,
                       label=fr'Bloch-like (ψ≈±0.5) — overall: {bloch_pct_overall:.1f}%'),
        ] + list(reversed(per_eta_handles))
        # ax[2].legend(handles=legend_handles, fontsize=15, loc='upper left',
        #              bbox_to_anchor=(1.02, 1.0), borderaxespad=0)

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ ({tag})")

    plt.tight_layout()
    plt.show()
    return cfg




for sample, coords in crop1.items():
    print (sample)

    for coord, data_dict in coords.items():
        vis.calc_neel_bloch_stats(crop1[sample][coord]['dw'], normalized=True)
        # calc_neel_bloch_stats(crop1[sample][coord]['dw'], normalized=False)

        plot_psi_bin_offset_w_highlight(crop1[sample][coord]['dw'], select_psi='', normalized=True, n_bins=11, offset=1.5)
        # plot_psi_bin_offset_w_highlight(crop1[sample][coord]['dw'], select_psi='', normalized=False, n_bins=11, offset=0.03)


# Compare Samples

In [ ]:



for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.calc_neel_bloch_stats(samples[sample][coord], normalized=True)
        # vis.calc_neel_bloch_stats(samples[sample][coord], normalized=False)

        vis.plot_psi_bin_offset_w_highlight(samples[sample][coord], select_psi='', normalized=True, n_bins=11, offset=2.5)



vis.plot_neel_bloch_comparison(samples, sample_ids, normalized=True)
# vis.plot_neel_bloch_comparison(samples, sample_ids, normalized=False)


# Calculate spatial energy variaton 

In [ ]:
def plot_energies_no_offset(cfg, sample, coord='y', normalized=True, z_indices=None):
    tag = 'normalized' if normalized else 'un-normalized'
    z_grid = cfg['z_grid']

    suffix = '' if normalized else '_raw'


    if normalized:
        energy_keys = ['pma_grid', 'exchange_grid', 'dmi_grid']
    else:
        energy_keys = ['pma_raw_grid', 'exchange_raw_grid', 'dmi_raw_grid']

    energy_labels = [r'PMA', r'Exchange', r'DMI']

    fig, ax = plt.subplots(3, 1, figsize=(10, 21))

    if z_indices is None:
        z_indices = np.linspace(0, cfg['n_rows'] - 1, cfg['n_rows']).astype(int)
    line_colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    linestyle = '-'

    for c, j in zip(line_colors, z_indices):

        for a, key in zip(ax, energy_keys):

            values = cfg[key][j, :]
            a.plot(x_data, values, color=c, linestyle=linestyle, marker = 'o', lw=1.5, zorder=2)


    for a, label in zip(ax, energy_labels):
        a.set_xlabel('x (pix)')
        a.set_ylabel('Energy density (a.u.)')
        a.set_title(label)
    

    fig.suptitle(f'Calculated Energy Densities Across Slice ({tag}) \n {sample} {coord}-slice')
    plt.tight_layout()
    plt.show()
    return cfg

def plot_energies_sample_comparison(cfg_dict, coord='y', normalized=True, z_indices=None,
                                     sample_colors=None, sample_thickness = None):
    """
    Compares energy densities across multiple samples (e.g. '452', '452_orig',
    '452_magconst') for each z-voxel, one figure PER z-voxel.

    cfg_dict: {sample_name: cfg} -- e.g. {'452': samples['452'][coord],
        '452_orig': samples['452_orig'][coord], '452_magconst': samples['452_magconst'][coord]}
        All cfgs are assumed to share the same coord/crop, so row index j
        refers to "the same" z-voxel across samples. If sample cfgs have
        different n_rows, only the indices common to all of them are used.

    sample_colors: dict {sample_name: color}. Defaults to
        {name: color} cycling through ['orange', 'purple', 'maroon'] in
        cfg_dict's insertion order if not provided.
    """
    tag = 'normalized' if normalized else 'un-normalized'
    suffix = '' if normalized else '_raw'

    if normalized:
        energy_keys = ['pma_grid', 'exchange_grid', 'dmi_grid']
    else:
        energy_keys = ['pma_raw_grid', 'exchange_raw_grid', 'dmi_raw_grid']
    energy_labels = [r'PMA', r'Exchange', r'DMI']



    # use only the z-voxel indices common to every sample being compared
    n_rows_all = [cfg['n_rows'] for cfg in cfg_dict.values()]
    n_rows_common = min(n_rows_all)
    if z_indices is None:
        z_indices = np.arange(n_rows_common)
    else:
        z_indices = z_indices[z_indices < n_rows_common]

    if len(set(n_rows_all)) > 1:
        print(f"Note: samples have differing n_rows {dict(zip(cfg_dict.keys(), n_rows_all))} "
              f"— only comparing the first {n_rows_common} shared z-voxel indices.")

    for j in z_indices:
        fig, ax = plt.subplots(3, 1, figsize=(5, 11))

        for sample, cfg in cfg_dict.items():
            color = sample_colors[sample]
            thick = sample_thickness[sample]
            x_data = cfg['unique_x'] - cfg['unique_x'].mean()
            z_val = cfg['z_grid'][j]

            for a, key in zip(ax, energy_keys):
                values = cfg[key][j, :]
                a.plot(x_data, values, color=color, linewidth = thick, linestyle='-', marker='o',
                       zorder=2, label=sample)

        for a, label in zip(ax, energy_labels):
            a.set_xlabel('x (pix)')
            a.set_ylabel('Energy density (a.u.)')
            a.set_title(label)
            a.legend(fontsize=9, loc='upper right', title='sample')

        fig.suptitle(f'Sample Comparison at z voxel = {j} ({tag}) \n {coord}-slice')
        plt.tight_layout()
        plt.show()

def plot_energies(cfg, sample, coord='y', offset=None, normalized=True, z_indices=None,
                   select_psi='', n_bins=11, psi_tol=0.05):
    """
    cfg is the grid-dict that calc_energies() was run on directly (e.g.
    crop1[sample][coord]['dw']) -- not a nested samples/crop1 wrapper.
    `sample`/`coord` are only used for the figure title.

    Each point on every energy line is now colored by its psi value (same
    twilight_shifted binning as plot_psi_bin_offset_w_highlight), with
    Neel-like (psi~0,+-1) and Bloch-like (psi~+-0.5) points overridden to
    cyan/magenta respectively.
    """
    tag = 'normalized' if normalized else 'un-normalized'
    z_grid = cfg['z_grid']

    suffix = '' if normalized else '_raw'
    psi_key = select_psi + f'psi{suffix}_grid'
    if psi_key not in cfg:
        cfg = vis.calc_psi(cfg)
    psi_over_pi = cfg[psi_key] / np.pi

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    psi_cmap = plt.cm.twilight_shifted
    bin_colors = psi_cmap(np.linspace(0, 1, n_bins))

    if normalized:
        energy_keys = ['pma_grid', 'exchange_grid', 'dmi_grid']
    else:
        energy_keys = ['pma_raw_grid', 'exchange_raw_grid', 'dmi_raw_grid']
    offset_shift = [1, 100, 4]
    energy_labels = [r'PMA', r'Exchange', r'DMI']

    fig, ax = plt.subplots(3, 1, figsize=(10, 21))

    if z_indices is None:
        z_indices = np.linspace(0, cfg['n_rows'] - 1, cfg['n_rows']).astype(int)
    line_colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    linestyle = '-'

    for c, j in zip(line_colors, z_indices):
        psi_slice = psi_over_pi[j, :]

        # bin each point by its psi value, then override Neel/Bloch points
        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx].copy()

        neel_mask = (np.abs(psi_slice) <= psi_tol) | (np.isclose(np.abs(psi_slice), 1, atol=psi_tol))
        bloch_mask = np.isclose(np.abs(psi_slice), 0.5, atol=psi_tol)
        point_colors[neel_mask] = mcolors.to_rgba('cyan')
        point_colors[bloch_mask] = mcolors.to_rgba('magenta')

        for a, key, ct in zip(ax, energy_keys, enumerate(offset_shift)):

            # print(offset_shift[ct[0]])
            shift = j * (offset if offset is not None else 0) / offset_shift[ct[0]]
            a.axhline(shift, color=c, lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'z voxel ={z_grid[j]:.1f}', xy=(0.26, shift - (0.15 / offset_shift[ct[0]])),
                       xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color=c)

            values = cfg[key][j, :] + shift
            a.plot(x_data, values, color=c, linestyle=linestyle, lw=1.5, zorder=2)
            a.scatter(x_data, values, c=point_colors, s=20, zorder=3)

    for a, label in zip(ax, energy_labels):
        a.set_xlabel('x (pix)')
        a.set_ylabel('Energy density (a.u.)')
        a.set_title(label)
        if offset is not None:
            a.set_yticks([])

    fig.suptitle(f'Calculated Energy Densities Across Slice ({tag}) \n {sample} {coord}-slice')
    plt.tight_layout()
    plt.show()
    return cfg

In [ ]:
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        crop1[sample][coord]['dw'] = vis.calc_energies(crop_info['dw'], coord, A=1.0, D=1.0, K=1.0)

In [ ]:
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        plot_energies_no_offset(crop_info['dw'], sample, coord=coord, normalized=True)
        # plot_energies(crop_info['dw'], sample, coord=coord, offset=1, normalized=False)

In [ ]:
# cfg_dict = {
#     '452': crop1['452']['y']['dw'],
#     '452_orig': crop1['452_orig']['y']['dw'],
#     '452_magconst': crop1['452_magconst']['y']['dw'],
# }

cfg_dict = {
    '270': crop1['270']['y']['dw'],
    '270_magconst': crop1['270_magconst']['y']['dw'],
}



default_palette = ['purple', 'orange']
thickness = [2,2]
colors = {name: default_palette[i % len(default_palette)]
                    for i, name in enumerate(cfg_dict.keys())}
thickness = {name: thickness[i % len(thickness)]
                    for i, name in enumerate(cfg_dict.keys())}
plot_energies_sample_comparison(cfg_dict, coord='y', normalized=True, sample_colors = colors,sample_thickness= thickness )


In [ ]:
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        plot_energies(crop_info['dw'], sample, coord=coord, offset=1, normalized=True)
        # plot_energies(crop_info['dw'], sample, coord=coord, offset=1, normalized=False)

oh great, thank you! Now, for the exchange curves saved as exchange_grid in the dictionary, please calculate the area under the curve and the inflection point(where divergence = 0) per z voxel. then, in two subplots, please plot area under the curve per z and in the second subplot plot (inflection point location per z voxel). Each subplot should have each samples data that is clearly labeled and offset from each other (like as is done above in plot_neel_bloch_comparison). use the orange maroon purple color scheme per sample. thank you! Please ask questions if needed.

In [ ]:

def find_inflection_point(x_data, curve, prominence=None):
    '''
    For a curve like (grad)**2 that does not cross zero, the inflection
    point of the underlying sigmoid corresponds to a local maximum of the
    curve (since |gradient| peaks at the inflection point).

    Finds all local maxima via scipy.signal.find_peaks, then selects the
    one closest to the center of x_data (handles cases where secondary
    peaks/shoulders exist away from the feature of interest).

    Sub-pixel location is refined via parabolic interpolation around the
    chosen peak.

    Returns np.nan if no local maximum is found.
    '''
    peaks, _ = find_peaks(curve, prominence=prominence)

    if len(peaks) == 0:
        return np.nan

    if len(peaks) > 1:
        center = np.mean(x_data)
        idx = peaks[np.argmin(np.abs(x_data[peaks] - center))]
    else:
        idx = peaks[0]

    x_peak = x_data[idx]

    # # parabolic sub-pixel refinement
    # if 0 < idx < len(curve) - 1:
    #     y0, y1, y2 = curve[idx - 1], curve[idx], curve[idx + 1]
    #     denom = (y0 - 2 * y1 + y2)
    #     if denom != 0:
    #         offset = 0.5 * (y0 - y2) / denom
    #     else:
    #         offset = 0.0
    #     dx = x_data[idx + 1] - x_data[idx]
    #     x_peak = x_data[idx] + offset * dx
    # else:
    #     x_peak = x_data[idx]

    return x_peak


def analyze_exchange_grid(cfg, normalized=True, z_indices=None):
    '''
    Computes the area under the curve (via np.trapezoid) and the inflection
    point (zero-crossing of the second derivative) for each z_voxel row of
    the exchange grid, and appends both results to cfg.
    Update the area method if trapezoidal approximation error is too high.
    '''

    area = []
    inflection = []

    suffix = '' if normalized else '_raw'
    energy_key = 'exchange_grid' if normalized else 'exchange_raw_grid'

    if z_indices is None:
        z_indices = np.arange(cfg['n_rows'])

    x_data = cfg['unique_x'] - cfg['unique_x'].mean()

    for j in z_indices:
        grid = cfg[energy_key][j, :]

        area_trap = integrate.trapezoid(grid, x=x_data)
        area.append(area_trap)

        x_inflect = find_inflection_point(x_data, grid)
        inflection.append(x_inflect)

    cfg[f'Exchange_area{suffix}'] = area
    cfg[f'Exchange_inflection{suffix}'] = inflection

    return cfg


for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        crop_info['dw'] = analyze_exchange_grid(crop_info['dw'], normalized=True)

In [ ]:
def plot_exchange_area_and_inflection(crop1, normalized=True, sample_colors = None, sample_thickness = 'None'):
    '''
    For each coord, create one figure with two subplots:
      (1) area vs z, for each sample
      (2) inflection point location vs z, for each sample

    Assumes analyze_exchange_grid() has already been run on each
    crop_info['dw'] dict, populating 'Exchange_area' / 'Exchange_inflection'
    (or their '_raw' counterparts).
    '''

    suffix = '' if normalized else '_raw'
    area_key = f'Exchange_area{suffix}'
    inflection_key = f'Exchange_inflection{suffix}'

    # palette = ['purple', 'orange', 'maroon']
    sample_names = list(crop1.keys())
    # sample_colors = {name: palette[i % len(palette)] for i, name in enumerate(sample_names)}

    # thickness = [4, 1, 1.5]
    # sample_thickness = {name: thickness[i % len(thickness)] for i, name in enumerate(sample_names)}


    # collect all unique coords across samples
    all_coords = set()
    for sample, coords in crop1.items():
        all_coords.update(coords.keys())

    figures = {}

    for coord in sorted(all_coords):
        fig, ax = plt.subplots(2, 1, figsize=(10, 14))

        for sample in sample_names:
            if coord not in crop1[sample]:
                continue

            cfg = crop1[sample][coord]['dw']

            if area_key not in cfg or inflection_key not in cfg:
                print(f'Warning: {sample}/{coord} missing {area_key} or {inflection_key}; skipping.')
                continue

            z_vals = cfg['z_grid'] - cfg['z_grid'][0]
            area = cfg[area_key]
            inflection = cfg[inflection_key]

            color = sample_colors[sample]
            thick = sample_thickness[sample]

            ax[0].plot(z_vals, area, color=color, label=sample, linewidth = thick, marker='o')
            ax[1].plot(z_vals, inflection, color=color, label=sample, linewidth = thick, marker='o')

        ax[0].set_xlabel('z voxel (bottom to top)')
        ax[0].set_ylabel('Area')
        ax[0].set_title(f'Area under Exchange vs z \n  {coord}-slice')
        ax[0].legend()

        ax[1].set_xlabel('z voxel (bottom to top)')
        ax[1].set_ylabel('Inflection point location')
        ax[1].set_title(f'Inflection Point vs z \n {coord}-slice')
        ax[1].legend()

        fig.tight_layout()
        figures[coord] = fig

    return figures


figs = plot_exchange_area_and_inflection(crop1, normalized=True,sample_colors = colors, sample_thickness = thickness)

# simulate walls

In [ ]:
### --- Define wall types and their parameters --- ###
# NOTE: these are idealized theoretical Bloch/Neel/Hybrid profiles (M/Ms),
# not derived from the measured data, so they are unaffected by the
# normalized-vs-un-normalized distinction used elsewhere in this notebook.
delta_demo = 2.5

xmin, xmax = 485, 515
x0 = np.linspace(0, (xmax - xmin), (xmax - xmin) + 1)
x0 = x0 - x0.mean()  # recenter around 0

wall_types = {
    'bloch':  {'psi': np.pi / 2, 'color': 'magenta'},
    'neel':   {'psi': 0,          'color': 'cyan'},
    'hybrid': {'psi': np.linspace(-np.pi /1.05, np.pi /1.1, 11), 'cmap': plt.cm.twilight_shifted},
}

for wall, cfg in wall_types.items():
    psi = cfg['psi']
    if np.isscalar(psi):
        Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi)
        cfg['Mx'], cfg['My'], cfg['Mz'] = Mx, My, Mz

        dx0 = x0[1] - x0[0]
        dMz_dx = np.gradient(Mz, dx0)
        dMy_dx = np.gradient(My, dx0)
        dMx_dx = np.gradient(Mx, dx0)
        cfg['dMx_dx'] = dMx_dx
        cfg['dMy_dx'] = dMy_dx
        cfg['dMz_dx'] = dMz_dx

    else:
        # hybrid: sweep over multiple psi values, store one array per psi
        Mx_list, My_list, Mz_list = [], [], []
        dMx_dx_list, dMy_dx_list, dMz_dx_list = [], [], []
        dx0 = x0[1] - x0[0]
        for psi_val in psi:
            Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi_val)
            Mx_list.append(Mx)
            My_list.append(My)
            Mz_list.append(Mz)
            dMz_dx_list.append(np.gradient(Mz, dx0))
            dMy_dx_list.append(np.gradient(My, dx0))
            dMx_dx_list.append(np.gradient(Mx, dx0))

        cfg['Mx'], cfg['My'], cfg['Mz'] = Mx_list, My_list, Mz_list
        cfg['dMx_dx'], cfg['dMy_dx'], cfg['dMz_dx'] = dMx_dx_list, dMy_dx_list, dMz_dx_list
        cfg['colors'] = cfg['cmap'](np.linspace(0, 1, len(psi)))


In [ ]:
for wall, cfg in wall_types.items():
    vis.plot_wall_components(wall, cfg, x0)
    vis.plot_wall_gradients(wall, cfg, x0)


# Segment Domain Walls

In [ ]:
def plot_dw_segments(cfg, walls, field='mzn_grid', grad_field=None, z_indices=None, offset=None):
    """
    Same stacked-offset line-cut style as plot_dw_segment, but with each
    detected domain wall's center marked as a short red dashed vertical
    line, confined to that row's own offset band (rather than spanning
    the whole plot height).
    """
    x_data = cfg['unique_x']
    z_grid = cfg['z_grid']
    profile_grid = cfg[field]

    if grad_field is None:
        grad_field = 'd' + field.replace('_grid', '') + '_dx'
    grad_grid = cfg[grad_field]

    if offset is None:
        offset = 0.8 if 'n_grid' in field else 0.5

    if z_indices is None:
        z_indices = np.arange(cfg['n_rows'])
    colors = plt.cm.viridis(np.linspace(0, 1, len(z_indices)))

    fig, ax = plt.subplots(1, 2, figsize=(15, 7))

    for c, row in zip(colors, z_indices):
        shift = row * offset

        ax[0].plot(x_data, profile_grid[row, :] + shift, color=c, lw=1.5, marker='o')
        ax[1].plot(x_data, grad_grid[row, :] + shift, color=c, lw=1.5, marker='o')

        for a in ax:
            a.axhline(shift, color=c, lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'z voxel ={z_grid[row]:.1f}', xy=(0.22, shift - 0.4),
                       xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=10, color=c)

        # mark every detected wall center for this row, on both panels
        half_height = offset * 0.4  # short vertical extent, confined to this row's band
        for w_idx, w_rows in walls.items():
            if row in w_rows:
                center_x = w_rows[row]['center_x']
                for a in ax:
                    a.plot([center_x, center_x], [shift - half_height, shift + half_height],
                           color='red', linestyle='--', lw=1.5, zorder=4)

    ax[0].set_ylabel(field)
    ax[0].set_xlabel('x (pix)')
    ax[0].set_title(f"{cfg['coord']} slice, {field}\n{cfg['sample']}")

    ax[1].set_xlabel('x (pix)')
    ax[1].set_ylabel(grad_field)
    ax[1].set_title(f"{cfg['coord']} slice, {grad_field}\n{cfg['sample']}")

    for a in ax:
        a.set_yticks([])

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def segment_domain_walls_voxel(
    cfg,
    field='mzn_grid',
    grad_field=None,
    height=None,
    prominence=0.1,
    distance=None,
    plot=False,
    z_indices=None,
    offset=None,
    min_coverage=0.50,
    max_shift_cols=3,
    max_gap_rows=1,
    require_contiguous=True,
):
    """
    Detect domain walls from peaks in |d(field)/dx| and track each physical
    wall through the z-stack by its x position.

    Walls are retained only if they occur in at least `min_coverage`
    of the full sample depth.

    Each retained z-voxel segment contains its original array-column bounds
    (`left_col`, `right_col`) so the same region can later be used to fit
    mzn_grid, mz_grid, or other fields.

    Returns
    -------
    walls : dict
        walls[wall_idx][row] = {
            'x',
            'mz',
            'grad',
            'center_x',
            'center_col',
            'left_col',
            'right_col',
            'z_voxel',
            'row'
        }

    flagged_walls : dict
        Diagnostic information for accepted but incomplete walls.
    """

    if grad_field is None:
        grad_field = 'd' + field.replace('_grid', '') + '_dx'

    x_data = np.asarray(cfg['unique_x'])
    z_grid = np.asarray(cfg['z_grid'])

    profile_grid = np.asarray(cfg[field])
    grad_grid = np.asarray(cfg[grad_field])

    n_rows = profile_grid.shape[0]

    # ================================================================
    # 1. DETECT ALL GRADIENT PEAKS IN EACH Z ROW
    # ================================================================

    detections_by_row = {}

    for row in range(n_rows):

        grad_row = grad_grid[row, :]
        prof_row = profile_grid[row, :]

        # Only search where gradient data are valid
        valid_mask = np.isfinite(grad_row)
        valid_idx = np.where(valid_mask)[0]

        if len(valid_idx) < 3:
            detections_by_row[row] = []
            continue

        grad_valid = grad_row[valid_idx]

        peaks, properties = find_peaks(
            np.abs(grad_valid),
            height=height,
            prominence=prominence,
            distance=distance,
        )

        if len(peaks) == 0:
            detections_by_row[row] = []
            continue

        # Convert peak indices back to original grid columns
        peak_cols = np.sort(valid_idx[peaks])

        row_left_edge = int(valid_idx.min())
        row_right_edge = int(valid_idx.max())

        row_detections = []

        # ------------------------------------------------------------
        # Build one segmented region around every detected peak
        # ------------------------------------------------------------
        for peak_number, center_col in enumerate(peak_cols):

            center_col = int(center_col)

            # Left boundary = halfway toward previous wall,
            # or edge of valid data for first wall
            if peak_number == 0:
                left_bound = row_left_edge
            else:
                left_bound = int(
                    (
                        peak_cols[peak_number - 1]
                        + center_col
                    ) // 2
                )

            # Right boundary = halfway toward next wall,
            # or edge of valid data for final wall
            if peak_number == len(peak_cols) - 1:
                right_bound = row_right_edge
            else:
                right_bound = int(
                    (
                        center_col
                        + peak_cols[peak_number + 1]
                    ) // 2
                )

            seg_slice = slice(
                left_bound,
                right_bound + 1
            )

            detection = {
                # ----------------------------------------------------
                # Segmented data
                # ----------------------------------------------------
                'x': x_data[seg_slice].copy(),
                'mz': prof_row[seg_slice].copy(),
                'grad': grad_row[seg_slice].copy(),

                # ----------------------------------------------------
                # Gradient-derived wall center
                # ----------------------------------------------------
                'center_x': x_data[center_col],
                'center_col': center_col,

                # ----------------------------------------------------
                # NEW:
                # Preserve the exact original grid columns used for
                # this wall segment.
                #
                # These are what the tanh fitting function should use
                # later to pull the equivalent region from mz_grid,
                # mzn_grid, etc.
                # ----------------------------------------------------
                'left_col': left_bound,
                'right_col': right_bound,

                # ----------------------------------------------------
                # Depth information
                # ----------------------------------------------------
                'z_voxel': z_grid[row],
                'row': int(row),
            }

            row_detections.append(detection)

        detections_by_row[row] = row_detections

    # ================================================================
    # 2. TRACK PHYSICAL WALLS THROUGH DEPTH
    # ================================================================
    #
    # A track represents ONE physical wall.
    #
    # We associate detections based on x-position rather than using
    # left-to-right peak number independently at every z row.
    # ================================================================

    tracks = {}
    next_track_id = 0

    for row in range(n_rows):

        detections = detections_by_row.get(row, [])

        if len(detections) == 0:
            continue

        # ------------------------------------------------------------
        # Find wall tracks that have been seen recently enough to
        # potentially continue into this row.
        # ------------------------------------------------------------
        active_tracks = []

        for track_id, track_rows in tracks.items():

            last_row = max(track_rows.keys())

            # max_gap_rows = 1 allows:
            #
            # row 4 -> missing row 5 -> reappear row 6
            #
            if row - last_row <= max_gap_rows + 1:

                last_detection = track_rows[last_row]

                active_tracks.append({
                    'track_id': track_id,
                    'last_row': last_row,
                    'center_col':
                        last_detection['center_col'],
                })

        # ------------------------------------------------------------
        # Generate every allowed track <-> detection match
        # ------------------------------------------------------------
        possible_matches = []

        for track_info in active_tracks:

            track_id = track_info['track_id']
            previous_col = track_info['center_col']

            for det_idx, detection in enumerate(detections):

                current_col = detection['center_col']

                dx_cols = abs(
                    current_col - previous_col
                )

                if dx_cols <= max_shift_cols:

                    possible_matches.append(
                        (
                            dx_cols,
                            track_id,
                            det_idx,
                        )
                    )

        # ------------------------------------------------------------
        # Match closest walls first
        # ------------------------------------------------------------
        possible_matches.sort(
            key=lambda item: item[0]
        )

        used_tracks = set()
        used_detections = set()

        for dx_cols, track_id, det_idx in possible_matches:

            if track_id in used_tracks:
                continue

            if det_idx in used_detections:
                continue

            # Assign this detection to the existing physical wall
            tracks[track_id][row] = detections[det_idx]

            used_tracks.add(track_id)
            used_detections.add(det_idx)

        # ------------------------------------------------------------
        # Any unmatched detection starts a new physical wall track
        # ------------------------------------------------------------
        for det_idx, detection in enumerate(detections):

            if det_idx in used_detections:
                continue

            tracks[next_track_id] = {
                row: detection
            }

            next_track_id += 1

    # ================================================================
    # 3. HELPER: LONGEST CONTIGUOUS DEPTH RUN
    # ================================================================

    def longest_consecutive_run(rows):

        if len(rows) == 0:
            return 0

        rows = sorted(rows)

        longest = 1
        current = 1

        for previous_row, current_row in zip(
            rows[:-1],
            rows[1:]
        ):

            if current_row == previous_row + 1:

                current += 1
                longest = max(
                    longest,
                    current
                )

            else:
                current = 1

        return longest

    # ================================================================
    # 4. FILTER WALLS ACCORDING TO DEPTH COVERAGE
    # ================================================================

    walls = {}
    flagged_walls = {}

    required_rows = int(
        np.ceil(
            min_coverage * n_rows
        )
    )

    print(
        f"\nWall filtering: requiring at least "
        f"{required_rows}/{n_rows} rows "
        f"({min_coverage * 100:.0f}% of sample)"
    )

    accepted_wall_idx = 0

    for track_id, track_rows in tracks.items():

        rows_present = sorted(
            track_rows.keys()
        )

        n_present = len(rows_present)

        coverage = (
            n_present / n_rows
        )

        longest_run = longest_consecutive_run(
            rows_present
        )

        contiguous_coverage = (
            longest_run / n_rows
        )

        # ------------------------------------------------------------
        # Test 1:
        # Is the wall present in at least 50% of all z voxels?
        # ------------------------------------------------------------
        passes_total = (
            n_present >= required_rows
        )

        # ------------------------------------------------------------
        # Test 2:
        # Does it continuously pass through at least 50% of sample?
        # ------------------------------------------------------------
        if require_contiguous:

            passes_contiguous = (
                longest_run >= required_rows
            )

        else:

            passes_contiguous = True

        # ------------------------------------------------------------
        # Reject insufficient total coverage
        # ------------------------------------------------------------
        if not passes_total:

            print(
                f"DROPPING track {track_id}: "
                f"{n_present}/{n_rows} rows "
                f"({coverage * 100:.1f}%)"
            )

            continue

        # ------------------------------------------------------------
        # Reject disconnected wall detections
        # ------------------------------------------------------------
        if not passes_contiguous:

            print(
                f"DROPPING track {track_id}: "
                f"total coverage = "
                f"{coverage * 100:.1f}%, "
                f"but longest continuous run = "
                f"{longest_run}/{n_rows} "
                f"({contiguous_coverage * 100:.1f}%)"
            )

            continue

        # ------------------------------------------------------------
        # Wall passes filtering
        # ------------------------------------------------------------
        walls[accepted_wall_idx] = track_rows

        # ------------------------------------------------------------
        # Keep but flag walls that do not span every z voxel
        # ------------------------------------------------------------
        if n_present < n_rows:

            flagged_walls[accepted_wall_idx] = {
                'coverage':
                    coverage,

                'n_rows_present':
                    n_present,

                'n_rows_total':
                    n_rows,

                'rows_present':
                    rows_present,

                'longest_contiguous_run':
                    longest_run,

                'contiguous_coverage':
                    contiguous_coverage,

                'original_track_id':
                    track_id,
            }

            print(
                f"FLAGGED wall {accepted_wall_idx}: "
                f"{n_present}/{n_rows} rows "
                f"({coverage * 100:.1f}%), "
                f"longest continuous run = "
                f"{longest_run}/{n_rows}"
            )

        else:

            print(
                f"KEEPING wall {accepted_wall_idx}: "
                f"detected in all {n_rows} rows"
            )

        accepted_wall_idx += 1

    # ================================================================
    # 5. OPTIONAL PLOT
    # ================================================================

    if plot:

        plot_dw_segments(
            cfg,
            walls,
            field=field,
            grad_field=grad_field,
            z_indices=z_indices,
            offset=offset,
        )

    return walls, flagged_walls

In [ ]:
domain_walls = {}
flagged_by_sample = {}

for sample, coords in samples.items():
    domain_walls[sample] = {}
    flagged_by_sample[sample] = {}
    for coord, cfg in coords.items():
        walls, flagged = segment_domain_walls_voxel(cfg, field='mzn_grid', prominence=0.1, plot = True)
        domain_walls[sample][coord] = walls
        flagged_by_sample[sample][coord] = flagged

In [ ]:
def fit_dw_wall(wall, sample, coord, wall_idx, offset = 1.5, label='', plot=True):
    """
    Fits a hyperbolic tangent to every z-voxel's segment for ONE domain wall
    (i.e. one entry of domain_walls[sample][coord]).

    Saves fit results in two places:
      - back into each row's segment dict (wall[row]['fit_params'], etc.) --
        so every voxel's fit lives right alongside its raw segment data
      - as return values (z_voxels, dw_width, dw_center arrays) -- for
        convenient use building a summary dict across walls/samples

    Rows are processed in sorted key order, NOT assumed to be 0..n-1
    contiguous, since some rows may be missing this wall.
    """
    pixel_size_nm = 8  # voxel size in nm

    rows_sorted = sorted(wall.keys())
    n = len(rows_sorted)
    colors_local = plt.cm.viridis(np.linspace(0, 1, n))

    z_voxels, dw_width, dw_center = [], [], []

    fig = plt.figure(figsize=(21, 6))
    fig.suptitle(f'Hyperbolic Tan Fit for {sample} {coord}, wall {wall_idx} ({label})',
                 fontsize=20, y=1.0)
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.set_xlabel('cross section (nm)')
    ax1.set_ylabel(r'$M_z$')
    ax1.set_title('Line cuts and fits')

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.set_xlabel('z voxel')
    ax2.set_ylabel('DW width')
    ax2.set_title('Fit DW width')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_xlabel('z voxel')
    ax3.set_ylabel('DW center')
    ax3.set_title('Fit DW center')

    for j, row in enumerate(rows_sorted):
        seg = wall[row]
        x = seg['x']
        Mz = seg['mz']
        z_v = seg['z_voxel']

        shift = j * offset

        x_data = x - x.mean()
        x_nm = x_data * pixel_size_nm

        ax1.plot(x_nm, Mz + shift, color=colors_local[j], lw=1.5, marker='o', label=f'z={z_v:.1f}')

        hyperbolic_fit, hyperbolic_error = vis.fit_hyperbolic(x_nm, Mz)
        hyperbolic_plot = vis.hyperbolic(x_nm, *hyperbolic_fit)

        width = np.abs(hyperbolic_fit[2])
        center = hyperbolic_fit[1]

        ax1.plot(x_nm, hyperbolic_plot + shift, color=colors_local[j], lw=3, linestyle='--')

        # save fit results back into this row's own segment dict
        seg['fit_params'] = hyperbolic_fit
        seg['fit_error'] = hyperbolic_error
        seg['fit_x_nm'] = x_nm
        seg['fit_profile'] = hyperbolic_plot
        seg['dw_width'] = width
        seg['dw_center'] = center

        z_voxels.append(z_v)
        dw_width.append(width)
        dw_center.append(center)

    z_voxels = np.array(z_voxels)
    dw_width = np.array(dw_width)
    dw_center = np.array(dw_center)

    ax1.legend(fontsize=8, ncol=2, title='z voxel', loc='upper right')

    ax2.plot(z_voxels, dw_width, marker='o')
    ax2.axhline(np.mean(dw_width), color='red',
                label=f'mean \N{GREEK CAPITAL LETTER DELTA} = {np.mean(dw_width):.2f}')
    ax2.legend()

    ax3.plot(z_voxels, dw_center, marker='o')

    if not plot:
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

    return z_voxels, dw_width, dw_center

In [ ]:
dw_fits = {}

for sample, coords in samples.items():
    dw_fits[sample] = {}
    for coord, data_dict in coords.items():
        dw_fits[sample][coord] = {}
        walls = domain_walls[sample][coord]

        for wall_idx, wall in walls.items():
            z_voxels, dw_width, dw_center = fit_dw_wall(
                wall, sample, coord, wall_idx, label=coord, plot=True
            )
            dw_fits[sample][coord][wall_idx] = {
                'z_voxel': z_voxels,
                'dw_width': dw_width,
                'dw_center': dw_center,
            }

In [ ]:
print(dw_fits['270']['y'].keys())
print(dw_fits['270']['y'][0]['z_voxel'])
print(dw_fits['270']['y'][10]['dw_width'])



In [ ]:
def average_dw_width_by_z_coord(param_dict, z_len, sample,coord,fit_type='normalized'):
    """
    Average fitted domain-wall width over all detected walls at each z voxel.

    """
    print(coord)

    mean_widths_list = []
    std_widths_list = []
    mean_centers_list = []
    std_centers_list = []

    rows_sorted = sorted(param_dict.keys())
    print('rows sorted')
    print(rows_sorted)
    n = len(rows_sorted)

    for z in range(z_len):
        print(f'z = {z}')
        widths_list_perz = []
        centers_list_perz = []

        for j in rows_sorted:
            print(f'j = {j}')
            seg = param_dict[j]

            dw_width = seg['dw_width'][z]
            # print(f'dw_width = {dw_width}')
            dw_center = seg['dw_center'][z]

            widths_list_perz.append(float(dw_width))
            centers_list_perz.append(float(dw_center))

        print(widths_list_perz)
        mean_widths_list.append(np.mean(np.asarray(widths_list_perz)))
        mean_centers_list.append(np.mean(np.asarray(centers_list_perz)))
        std_widths_list.append(np.std(np.asarray(widths_list_perz)))
        std_centers_list.append(np.std(np.asarray(centers_list_perz)))

    # print(mean_widths_list)

    return mean_widths_list, std_widths_list, mean_centers_list, std_centers_list


dw_averages = {}

for sample, coords in samples.items():
    dw_averages[sample] = {}
    for coord, data_dict in coords.items():
        dw_averages[sample][coord] = {}
            
        z_len = len(samples[sample][coord]['z_grid'])
        print(z_len)
        average_width, error_width, average_center, error_center = average_dw_width_by_z_coord(dw_fits[sample][coord], z_len, sample,coord,fit_type='normalized')
        dw_averages[sample][coord] = {
            'width_z' : average_width, 
            'width_ze' : error_width, 
            'center_z' : average_center, 
            'center_ze' : error_center
        }



In [ ]:
def plot_width_z_dependence(dw_averages, colors): 
    fig, ax = plt.subplots(1,2, figsize=(12, 6))


    for sample, coords in samples.items():
        print(f'sample = {sample}')

        for coord in coords:

            if coord == 'x': 
                line = ':'
            else: 
                line = '--'
            width = np.array(dw_averages[sample][coord]['width_z'])
            width_e = np.array(dw_averages[sample][coord]['width_ze'])
            center = np.array(dw_averages[sample][coord]['center_z'])
            center_e = np.array(dw_averages[sample][coord]['center_ze'])

            z = np.linspace(0, len(width), len(width))

            label = (f'{sample}, {coord}')

            ax[0].errorbar(z, width, yerr = width_e, color = colors[sample], marker='o', linestyle=line, linewidth=1.5, capsize=3, label=label)
            ax[1].errorbar(z, abs(center), yerr = center_e, color = colors[sample], marker='o', linestyle=line, linewidth=1.5, capsize=3, label=label)

    ax[0].set_xlabel('Depth through sample')
    ax[0].set_ylabel('Average domain wall width (nm)')
    ax[0].set_title('Average Domain Wall Width vs Depth')
    ax[0].legend()
    ax[0].set_ylim(0,50)

    ax[1].set_xlabel('Depth through sample')
    ax[1].set_ylabel('Abs domain wall center')
    ax[1].set_title('Domain Wall Center vs Depth')
    ax[1].legend()

    plt.tight_layout()
    plt.show()

    return fig, ax

sample_colors = { '665' : 'gold', 
                 '270': 'deeppink',
                 '452': 'limegreen', 
                 '105': 'cyan'} 
plot_width_z_dependence(dw_averages, sample_colors)

In [ ]:
# def average_dw_width_by_z_coord(param_dict, z_len, sample,coord,fit_type='normalized'):
#     """
#     Average fitted domain-wall width over all detected walls at each z voxel.

#     """
#     print(coord)

#     mean_widths_list = []
#     std_widths_list = []
#     mean_centers_list = []
#     std_centers_list = []

#     rows_sorted = sorted(param_dict.keys())
#     print('rows sorted')
#     print(rows_sorted)
#     n = len(rows_sorted)

#     for z in range(z_len):
#         # print(f'z = {z}')
#         widths_list_perz = []
#         centers_list_perz = []

#         for j in rows_sorted:

#             seg = param_dict[j]

#             dw_width = seg['dw_width'][z]
#             # print(f'dw_width = {dw_width}')
#             dw_center = seg['dw_center'][z]

#             widths_list_perz.append(float(dw_width))
#             centers_list_perz.append(float(dw_center))

#         # print(widths_list_perz)
#         mean_widths_list.append(np.mean(np.asarray(widths_list_perz)))
#         mean_centers_list.append(np.mean(np.asarray(centers_list_perz)))
#         std_widths_list.append(np.std(np.asarray(widths_list_perz)))
#         std_centers_list.append(np.std(np.asarray(centers_list_perz)))

#     # print(mean_widths_list)

#     return mean_widths_list, std_widths_list, mean_centers_list, std_centers_list


# dw_averages = {}

# for sample, coords in samples.items():
#     dw_averages[sample] = {}
#     for coord, data_dict in coords.items():
#         dw_averages[sample][coord] = {}
            
#         z_len = len(samples[sample][coord]['z_grid'])
#         average_width, error_width, average_center, error_center = average_dw_width_by_z_coord(dw_fits[sample][coord], z_len, sample,coord,fit_type='normalized')
#         dw_averages[sample][coord] = {
#             'width_z' : average_width, 
#             'width_ze' : error_width, 
#             'center_z' : average_center, 
#             'center_ze' : error_center
#         }



# domain wall width

In [ ]:
# Functions
def fit_linecut_hyperbolic(x, Mz, z_shift, xmax, xmin, sample, label='', plot=True):
    pixel_size_nm = 8  # voxel size in nm
    x_data = x - x.mean()
    x_nm = x_data * pixel_size_nm

    colors_local = plt.cm.viridis(np.linspace(0, 1, len(z_shift)))

    dw_width = []
    dw_center = []

    fig = plt.figure(figsize=(21, 6))
    fig.suptitle(f'Hyperbolic Tan Fit of Data for range ({xmin}, {xmax}) \n {sample} ({label})', fontsize=20, y=1.0)
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.set_xlabel('cross section (nm)')
    ax1.set_ylabel(r'$M_z$' + (r'$/ M_{norm}$' if label == 'normalized' else ''))
    ax1.set_title('Line cuts of domains at different z')

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.set_xlabel('slice (%)')
    ax2.set_ylabel('DW width')
    ax2.set_title('fit DW width at different slices')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_xlabel('slice (%)')
    ax3.set_ylabel('DW center')
    ax3.set_title('fit DW center at different slices')

    for j, eta_v in enumerate(z_shift):
        # Linecut
        ax1.plot(x_nm, Mz[j, :], color=colors_local[j], lw=1.5, marker='o', label=fr'{eta_v*100:.0f}%')

        # Fit with hyperbolic tangent
        hyperbolic_fit, hyperbolic_error = vis.fit_hyperbolic(x_nm, Mz[j, :])
        hyperbolic_plot = vis.hyperbolic(x_nm, *hyperbolic_fit)
        dw_width.append(hyperbolic_fit[2])
        dw_center.append(hyperbolic_fit[1])

        ax1.plot(x_nm, hyperbolic_plot, color=colors_local[j], linewidth=3, linestyle='--')

    ax1.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper right')
    ax2.plot(z_shift*100, np.abs(dw_width), marker='o')
    ax2.axhline(np.mean(np.abs(dw_width)), color='red', label=f'mean \N{GREEK CAPITAL LETTER DELTA} = {np.mean(dw_width):.2f}')
    ax2.legend()
    ax3.plot(z_shift*100, dw_center, marker='o')

    if not plot:
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

    return np.abs(dw_width), dw_center


In [ ]:
# Fit the hyperbolic-tangent domain-wall profile for BOTH the normalized
# (mzn_grid) and un-normalized (mz_grid) Mz components, storing both under
# dw_fits[sample][coord]['normalized'] / ['raw'].
dw_fits = {}

for sample, coords in samples.items():
    dw_fits[sample] = {}
    for coord, data_dict in coords.items():

        wall = crop1[sample][coord]['dw']

        dw_width_n, dw_center_n = fit_linecut_hyperbolic(
            wall['unique_x'], wall['mzn_grid'], wall['eta_targets'],
            crop1[sample][coord]['xmax'], crop1[sample][coord]['xmin'],
            sample=sample, label='normalized', plot=True
        )
        dw_width_raw, dw_center_raw = fit_linecut_hyperbolic(
            wall['unique_x'], wall['mz_grid'], wall['eta_targets'],
            crop1[sample][coord]['xmax'], crop1[sample][coord]['xmin'],
            sample=sample, label='un-normalized', plot=True
        )

        dw_fits[sample][coord] = {
            'normalized': {'dw_width': dw_width_n, 'dw_center': dw_center_n},
            'raw': {'dw_width': dw_width_raw, 'dw_center': dw_center_raw},
            'eta_targets': wall['eta_targets'],
        }

print(dw_fits['665']['x'].keys())
print(dw_fits['665']['y'].keys())


In [ ]:
## Compare x and y slice
sample_colors = ['gold', 'deeppink', 'limegreen', 'cyan']

## Compare sample per x and y (normalized branch, the physically bounded [-1,1] Mz)
for coord in ['x', 'y']:
    fig, ax = plt.subplots(figsize=(10, 6))
    for sample, color in zip(samples, sample_colors):
        fit = dw_fits[sample][coord]
        ax.plot(fit['eta_targets'] * 100, fit['normalized']['dw_width'], color=color, marker='o', label=f'{sample}')

    ax.set_xlabel('slice (%)')
    ax.set_ylabel('DW width (nm)')
    ax.set_title(f'Fit DW width across samples, {coord}-slice (Mz, normalized)')
    ax.legend(loc='upper right', title='sample')
    plt.tight_layout()
    plt.show()


fit theta per x in thickness

In [ ]:
### we know that theta(x) = arccos(mz), computed once from the normalized mz
### (theta isn't itself split into normalized/un-normalized -- it's always
### derived from the normalized mz in get_slice(), so there's only one theta_grid)
def plot_theta(cfg, colors):
    fig, ax = plt.subplots(figsize=(7, 7))
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    for j, eta_v in enumerate(cfg['eta_targets']):
        ax.plot(x_data, cfg['theta_grid'][j, :]/np.pi, color=colors[j], marker='o', lw=1.5, label=fr'{eta_v*100:.0f}%')

    ax.set_ylabel(r'θ(x) = arccos($m_z$) (1/π)')
    ax.set_xlabel('x (pix)')
    ax.axhline(1, color='crimson', label='sin(θ) = 0!!')

    ax.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper left')
    ax.set_title(f"Calculated theta per x \n {cfg['sample']} {cfg['coord']}-slice")


for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        plot_theta(crop1[sample][coord]['dw'], colors)
